In [12]:
import boto3
from pprint import  pprint

lambda_client = boto3.client(
    "lambda",
    endpoint_url="http://localhost:4566",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1",
)

response = lambda_client.list_functions()
pprint(response)

{'Functions': [{'Architectures': ['x86_64'],
                'CodeSha256': 'VyiqNJmHqvADDQY8DsdsLQBnFbzlykBvP65CEvJNjmA=',
                'CodeSize': 366,
                'Description': '',
                'EphemeralStorage': {'Size': 512},
                'FunctionArn': 'arn:aws:lambda:us-east-1:000000000000:function:hello-lambda',
                'FunctionName': 'hello-lambda',
                'Handler': 'lambda_function.handler',
                'LastModified': '2026-07-30T09:46:46.801581+0000',
                'LoggingConfig': {'LogFormat': 'Text',
                                  'LogGroup': '/aws/lambda/hello-lambda'},
                'MemorySize': 128,
                'PackageType': 'Zip',
                'RevisionId': '704d6e80-94a4-46a4-aa1a-db2f9961af6e',
                'Role': 'arn:aws:iam::000000000000:role/lambda-role',
                'Runtime': 'python3.12',
                'SnapStart': {'ApplyOn': 'None', 'OptimizationStatus': 'Off'},
                'Timeout': 30,
 

In [7]:
import boto3
import json

iam = boto3.client(
    "iam",
    endpoint_url="http://localhost:4566",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1",
)

assume_role_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "lambda.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

response = iam.create_role(
    RoleName="lambda-role",
    AssumeRolePolicyDocument=json.dumps(assume_role_policy),
)

print(response["Role"]["Arn"])

arn:aws:iam::000000000000:role/lambda-role


In [8]:
import boto3

lambda_client = boto3.client(
    "lambda",
    endpoint_url="http://localhost:4566",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1",
)

with open("function.zip", "rb") as f:
    zip_bytes = f.read()

response = lambda_client.create_function(
    FunctionName="hello-lambda",
    Runtime="python3.12",
    Role="arn:aws:iam::000000000000:role/lambda-role",
    Handler="lambda_function.handler",
    Code={
        "ZipFile": zip_bytes
    },
    Timeout=30,        # optional
    MemorySize=128,    # optional
    Publish=True       # optional
)

print(response["FunctionArn"])

arn:aws:lambda:us-east-1:000000000000:function:hello-lambda


In [31]:
with open("function.zip", "rb") as f:
    zip_bytes = f.read()

lambda_client.update_function_code(
    FunctionName="hello-lambda",
    ZipFile=zip_bytes,
    Publish=True,
)

{'ResponseMetadata': {'RequestId': '2b2de31f-17bd-4c78-b478-a80c83c73e96',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'TwistedWeb/26.4.0',
   'date': 'Thu, 30 Jul 2026 10:07:03 GMT',
   'content-type': 'text/plain; charset=utf-8',
   'content-length': '1002',
   'x-amzn-requestid': '2b2de31f-17bd-4c78-b478-a80c83c73e96',
   'x-amz-request-id': '2b2de31f-17bd-4c78-b478-a80c83c73e96',
   'x-localstack': '2026.7.1'},
  'RetryAttempts': 0},
 'FunctionName': 'hello-lambda',
 'FunctionArn': 'arn:aws:lambda:us-east-1:000000000000:function:hello-lambda:7',
 'Runtime': 'python3.12',
 'Role': 'arn:aws:iam::000000000000:role/lambda-role',
 'Handler': 'lambda_function.handler',
 'CodeSize': 812,
 'Description': '',
 'Timeout': 30,
 'MemorySize': 128,
 'LastModified': '2026-07-30T10:07:03.051604+0000',
 'CodeSha256': 'KShVD/Kmp+8ifMcV+SIT/aHeC4Tnv3P2qu00+Wfc53k=',
 'Version': '7',
 'TracingConfig': {'Mode': 'PassThrough'},
 'RevisionId': '7f80f03b-bad8-4db7-8710-296e2306b96c',
 'State': '

In [32]:
import json

response = lambda_client.invoke(
    FunctionName="hello-lambda",
    Payload=json.dumps({"url": "pants-url-to-predict"}).encode(),
)

result = json.loads(response["Payload"].read())
print(result)

{'statusCode': 200, 'body': 'Hello pants-url-to-predict!'}
